## 1. Project setup




In [1]:
from pathlib import Path
import sys
import json

PROJECT_ROOT = Path.cwd()
if not ((PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT / 'data').exists()):
    possible = Path.cwd() / 'Personal_Python_QA_Chatbot_From_Scratch'
    if (possible / 'src').exists():
        PROJECT_ROOT = possible

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.python_chatbot import (
    tokenize, load_knowledge_base, build_tfidf, vectorize_query,
    cosine_similarity, PythonChatbot, create_default_chatbot
)

print('Project folder:', PROJECT_ROOT)
print('Implementation imported successfully.')

Project folder: /mnt/data/Personal_Python_QA_Chatbot_From_Scratch
Implementation imported successfully.


## 2. Load and inspect our knowledge base




In [2]:
knowledge_path = PROJECT_ROOT / 'data' / 'knowledge_base.json'
knowledge_base = load_knowledge_base(knowledge_path)

print('Number of knowledge-base entries:', len(knowledge_base))
for entry in knowledge_base[:5]:
    print(f"{entry['id']} | {entry['topic']}")
    print('Question:', entry['question'])
    print('Answer preview:', entry['answer'][:150] + '...')
    print('-' * 70)

Number of knowledge-base entries: 40
PY001 | What is Python?
Question: What is Python?
Answer preview: Python is a general-purpose programming language designed to make code readable. It is commonly used for automation, web development, data analysis, s...
----------------------------------------------------------------------
PY002 | Variables
Question: What is a variable in Python?
Answer preview: A variable is a name that refers to a value. You create or update a variable with assignment, for example: age = 20. Python determines the value's typ...
----------------------------------------------------------------------
PY003 | Python data types
Question: What are the common data types in Python?
Answer preview: Common built-in Python types include int for whole numbers, float for decimal numbers, str for text, and bool for True or False values. Python also pr...
----------------------------------------------------------------------
PY004 | Strings
Question: What is a string in Python?


## 3. Understand our tokenizer



In [3]:
sample_text = 'How do I create a Python function named calculate_total(value)?'
print('Original:', sample_text)
print('Tokens:  ', tokenize(sample_text))

Original: How do I create a Python function named calculate_total(value)?
Tokens:   ['create', 'python', 'function', 'named', 'calculate_total', 'value']


## 4. Build TF-IDF vectors



In [4]:
search_documents = [
    entry['question'] + ' ' + ' '.join(entry['alternate_questions']) + ' ' +
    ' '.join(entry['keywords']) + ' ' + entry['topic'] + ' ' + entry['answer']
    for entry in knowledge_base
]
document_vectors, idf_values = build_tfidf(search_documents)

print('Vocabulary size:', len(idf_values))
print('First 15 vocabulary terms:', list(idf_values)[:15])
print('First vector length:', len(document_vectors[0]))
print('First vector norm:', round(sum(v*v for v in document_vectors[0].values()) ** 0.5, 6))

Vocabulary size: 705
First 15 vocabulary terms: ['is', 'python', 'programming', 'language', 'people', 'give', 'introduction', 'interpreted', 'general', 'purpose', 'readable', 'designed', 'make', 'code', 'it']
First vector length: 38
First vector norm: 1.0


## 5. Compare a query with an entry




In [5]:
query = 'How is a tuple different from a list?'
query_vector = vectorize_query(query, idf_values)
example_index = next(i for i, entry in enumerate(knowledge_base) if entry['id'] == 'PY007')
score = cosine_similarity(query_vector, document_vectors[example_index])
print('Query:', query)
print('Matched entry:', knowledge_base[example_index]['topic'])
print('Cosine similarity:', round(score, 4))

Query: How is a tuple different from a list?
Matched entry: Lists versus tuples
Cosine similarity: 0.4078


## 6. Create the chatbot




In [6]:
chatbot = create_default_chatbot(threshold=0.18)
print('Chatbot is ready.')
print('Supported topics:')
for topic in chatbot.suggest_topics(limit=40):
    print('-', topic)

Chatbot is ready.
Supported topics:
- What is Python?
- Variables
- Python data types
- Strings
- Lists
- Tuples
- Lists versus tuples
- Dictionaries
- Sets
- If statements
- For loops
- While loops
- Break and continue
- Functions
- Parameters and arguments
- Lambda functions
- Recursion
- List comprehensions
- Object-oriented programming
- Classes and objects
- The init method
- Inheritance
- Exceptions
- Common Python errors
- File reading and writing
- Modules
- Packages
- Virtual environments
- Pip
- Range
- Enumerate
- Zip
- Sorting
- Map and filter
- Boolean operators
- Equality and identity
- Type conversion
- Comments
- Docstrings
- Main guard


## 7. Ask the chatbot a question



In [7]:
user_question = 'How can I handle errors when my program runs?'
result = chatbot.answer(user_question)

print('You:', user_question)
print('Chatbot:', result.answer)
print('Matched topic:', result.matched_topic)
print('Retrieval score:', result.confidence)
print('Fallback used:', result.is_fallback)

You: How can I handle errors when my program runs?
Chatbot: Exception handling lets a program respond to errors that occur while it runs. Put code that may fail inside try and handle selected error types with except. An optional else block runs when no exception occurred, and finally runs whether or not an exception occurred. Catch specific exceptions where possible instead of hiding every problem with a broad except.
Matched topic: Exceptions
Retrieval score: 0.3634
Fallback used: False


## 8. Try several questions




In [8]:
test_path = PROJECT_ROOT / 'data' / 'test_questions.json'
with test_path.open('r', encoding='utf-8') as handle:
    test_questions = json.load(handle)

for item in test_questions:
    response = chatbot.answer(item['question'])
    print('Question:', item['question'])
    print('Expected:', item['expected_topic'])
    print('Retrieved:', response.matched_topic)
    print('Fallback:', response.is_fallback)
    print('-' * 60)

Question: How is a tuple different from a list?
Expected: Lists versus tuples
Retrieved: Lists versus tuples
Fallback: False
------------------------------------------------------------
Question: Explain an anonymous function in Python
Expected: Lambda functions
Retrieved: Lambda functions
Fallback: False
------------------------------------------------------------
Question: How do I catch runtime errors?
Expected: Exceptions
Retrieved: Exceptions
Fallback: False
------------------------------------------------------------
Question: How can I get an index and item while looping?
Expected: Enumerate
Retrieved: Enumerate
Fallback: False
------------------------------------------------------------
Question: How do I save text to a file?
Expected: File reading and writing
Retrieved: File reading and writing
Fallback: False
------------------------------------------------------------
Question: What does __name__ equal when running a script?
Expected: Main guard
Retrieved: Main guard
Fallbac

## 9. A question outside the knowledge base




In [9]:
unknown_question = 'Who won the football world cup in 1998?'
unknown_result = chatbot.answer(unknown_question)
print('Question:', unknown_question)
print('Answer:', unknown_result.answer)
print('Fallback used:', unknown_result.is_fallback)

Question: Who won the football world cup in 1998?
Answer: I do not have a reliable answer for that question in my current Python knowledge base. Try rephrasing it or add the topic and its explanation to data/knowledge_base.json.
Fallback used: True


## 10. Simple retrieval evaluation




In [10]:
known_items = [item for item in test_questions if item['expected_topic'] is not None]
correct = 0
for item in known_items:
    response = chatbot.answer(item['question'])
    if response.matched_topic == item['expected_topic']:
        correct += 1

retrieval_accuracy = correct / len(known_items) if known_items else 0.0
unknown_items = [item for item in test_questions if item['expected_topic'] is None]
unknown_fallbacks = sum(chatbot.answer(item['question']).is_fallback for item in unknown_items)

print(f'Correct known-topic matches: {correct}/{len(known_items)}')
print(f'Known-topic retrieval accuracy: {retrieval_accuracy:.1%}')
print(f'Unknown questions correctly sent to fallback: {unknown_fallbacks}/{len(unknown_items)}')
print('Note: scores are results on this small manually prepared test set only.')

Correct known-topic matches: 6/6
Known-topic retrieval accuracy: 100.0%
Unknown questions correctly sent to fallback: 1/1
Note: scores are results on this small manually prepared test set only.
